## Analisis Exploratorio de los Datos (EDA) 
### Christian Pascual

---

1. **Carga de Librerías**

In [ ]:
# manipulación de datos
import pandas as pd

# visualización
import matplotlib.pyplot as plt
import seaborn as sns

# estilo de gráficos
sns.set(style="whitegrid")

# función para cargar datos
from cargar_datos import cargar_datos

---

2. **Exploración Inicial del Dataset**

In [ ]:
# cargar datos
df = cargar_datos()

# ver primeras filas
df.head()

In [ ]:
# dimensiones del dataset
df.shape

In [ ]:
# nombres de columnas
df.columns

In [ ]:
# nombres de columnas
df.columns

**Interpretación**

- El dataset contiene aprox. 10k registros y 23 variables.

- Existen variables numéricas, categóricas y de fecha.

- Algunas columnas presentan valores faltantes que deberán revisarse en el análisis de calidad de datos.

- La variable objetivo del problema es `Pago_atiempo`.

---

3. **Preparación del Dataset**

In [ ]:
# Creamos la función para poder preparar el dataset y se pueda reproducir en proximas etapas del PI
def dataset_preparado(df):

    # copiar dataframe
    df = df.copy()

    # representaciones de valores nulos
    valores_nulos = ["NA","N/A","na","null","None","","?","nan"]

    # reemplazar por NA
    df.replace(valores_nulos, pd.NA, inplace=True)

    # valores válidos tendencia
    valores_validos = ["Creciente","Decreciente","Estable"]

    # función para limpiar tendencia
    def limpiar_tendencia(valor):

        if pd.isna(valor):
            return pd.NA

        valor = str(valor).strip()

        if valor in valores_validos:
            return valor

        return pd.NA

    # limpiar columna
    df["tendencia_ingresos"] = df["tendencia_ingresos"].apply(limpiar_tendencia)

    # convertir tipos
    df["tipo_credito"] = df["tipo_credito"].astype("category")
    df["tipo_laboral"] = df["tipo_laboral"].astype("category")
    df["tendencia_ingresos"] = df["tendencia_ingresos"].astype("category")

    # convertir fecha
    df["fecha_prestamo"] = pd.to_datetime(df["fecha_prestamo"])

    # convertir target
    df["Pago_atiempo"] = df["Pago_atiempo"].astype("bool")

    return df

In [ ]:
# Cargamos el dataset al dataframe despues de aplicar la funcion de preparación
df = dataset_preparado(df)

**Interpretación**

- Se unificaron representaciones de valores nulos.

- Se corrigieron inconsistencias en tendencia_ingresos.

- Se ajustaron tipos de datos para variables categóricas y fechas.

- Esto permite realizar el EDA con tipos de datos consistentes.

---

4. **Calida de los Datos**

In [ ]:
# Valores Nulos

nulos = df.isnull().sum()

porcentaje_nulos = (nulos / len(df)) * 100

tabla_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": porcentaje_nulos
})

tabla_nulos.sort_values("porcentaje", ascending=False)

In [ ]:
# verificar duplicados
df.duplicated().sum()

**Interpretación**

- Algunas variables presentan porcentaje relevante de valores faltantes, especialmente:

    - `tendencia_ingresos`

    - `promedio_ingresos_datacredito`

- No se detectaron registros duplicados en el dataset.

- Los valores faltantes deberán analizarse posteriormente en FT_Engineering.

---

5. **Identificación de Variables**

In [ ]:
# variables numéricas
variables_numericas = df.select_dtypes(include=["int64","float64"]).columns

# variables categóricas
variables_categoricas = df.select_dtypes(include=["category"]).columns

# imprimir variables
print("Variables numéricas:\n", variables_numericas)
print("\nVariables categóricas:\n", variables_categoricas)

**Interpretación**

Interpretación

Tipos de variables identificados:

- **Numéricas**	              ->      `capital_prestado`, `salario_cliente`
- **Categóricas nominales**	  ->      `tipo_credito`, `tipo_laboral`
- **Ordinales**	              ->      `tendencia_ingresos`
- **Fecha**	                  ->      `fecha_prestamo`
- **Dicotómica**	          ->      `Pago_atiempo`

---

6. **Analisis Univariable**

- **Estadisticas**

In [ ]:
# estadísticas de variables numéricas
df[variables_numericas].describe()

- **Asimetría y Curtosis**

In [ ]:
# skewness
df[variables_numericas].skew()

# curtosis
df[variables_numericas].kurtosis()

- **Histogramas**

In [ ]:
# histogramas
df[variables_numericas].hist(
    figsize=(14,10),
    bins=30
)

plt.tight_layout()
plt.show()

- **Boxplots**

In [ ]:
# boxplots
for col in variables_numericas:

    plt.figure(figsize=(6,4))

    sns.boxplot(x=df[col])

    plt.title(col)

    plt.show()

**Interpretación**

- Varias variables financieras presentan distribuciones sesgadas.

- Se detectan outliers en montos financieros.

- Esto es común en datos económicos.

---

7. **Analisis Categórico**

In [ ]:
for col in variables_categoricas:

    print(df[col].value_counts())

    sns.countplot(x=col, data=df)

    plt.title(col)

    plt.xticks(rotation=45)

    plt.show()

**Interpretación**

- Se observan diferentes distribuciones entre tipos de crédito y tipo laboral.

- Algunas categorías tienen poca representación, lo que puede afectar el modelado.

---

 8. **Analisis Bivariable**

- **Distribución del target**

In [ ]:
# proporción del target
df["Pago_atiempo"].value_counts(normalize=True) * 100

In [ ]:
# gráfico del target
sns.countplot(
    x="Pago_atiempo",
    data=df
)

plt.title("Distribución del target")

plt.show()

**Interpretación**

- El dataset presenta fuerte desbalance de clases.

- Aproximadamente 95% de los clientes pagan a tiempo.

- **Numéricas vs Target**

In [ ]:
# variables numéricas vs target
for col in variables_numericas:

    plt.figure(figsize=(6,4))

    sns.boxplot(
        x="Pago_atiempo",
        y=col,
        data=df
    )

    plt.title(f"{col} vs Pago_atiempo")

    plt.show()

**Interpretación**

- Algunas variables financieras muestran diferencias entre clientes que pagan y los que no.

- Estas variables podrían ser predictoras relevantes para el modelo.

---

9. **Analisis Multivariable**

- **Matriz de Correlación**

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    df[variables_numericas].corr(),
    cmap="coolwarm"
)

plt.title("Matriz de correlación")

plt.show()

- **Pairplot**

In [ ]:
sns.pairplot(
    df[variables_numericas].sample(1000)
)

**Interpretación**

- Existen correlaciones entre variables financieras.

- Algunas variables podrían contener información redundante.

---

10. **Reglas de validación de datos**

A partir del EDA se identificaron posibles reglas:

- `edad_cliente` >= 18

- `capital_prestado` > 0

- `salario_cliente` >= 0

- `plazo_meses` > 0

- `Pago_atiempo` = {*True*, *False*}

- `tendencia_ingresos` = {*Creciente*, *Decreciente*, *Estable*}

---

11. **Posibles transformaciones futuras**

En la etapa de Feature Engineering podrían aplicarse:

- imputación de valores faltantes

- normalización o escalado

- encoding de variables categóricas

- manejo de outliers

---

12. Variables derivadas potenciales

Posibles variables útiles:

- **_ratio deuda / ingresos_**

    *`saldo_total` / `salario_cliente`*

- **_cuota / ingreso_**

    *`cuota_pactada` / `salario_cliente`*

- Estas variables pueden mejorar la capacidad predictiva del modelo.

---

13. **Conclusiones**

Se llegaron a las siguientes conclusiones en este avance:

- Dataset financiero con **_10k registros_**.

- Variables numéricas dominan el dataset.

- Se identificaron **_valores faltantes importantes (nulos)_**.

- Existe **_fuerte desbalance en la variable objetivo (95% vs 5%)_**.

- Algunas variables financieras podrían ser **_predictoras relevantes del pago del crédito_**.